### Middleware

In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv('OPENAI_API_KEY')
os.environ["GEMINI_API_KEY"]=os.getenv('GEMINI_API_KEY')
os.environ["GROQ_API_KEY"]=os.getenv('GROQ_API')

### Summarization 

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
agent = create_agent(
    model="gpt-4o", 
    checkpointer=InMemorySaver(), 
    middleware=[
        SummarizationMiddleware(
        model="gpt-4o",
        trigger=("messages", 10),
        keep=("messages", 4)
        )
    ]
)

In [15]:
config = {"configurable": {"thread_id": "test-1"}}

In [16]:
question=[
"What is 5+7?",
"What is 12-4?",
"What is 36?",
"What is 20/4?",
"What is 15+9?",
"What is 87?"
]
for q in question:
    res = agent.invoke({"messages":[HumanMessage(content=q)]}, config=config)
    print(f"Messages: {res}")
    print(f"Messsges:{len(res["messages"])}")
    for message in res["messages"]:
        print(message)

Messages: {'messages': [HumanMessage(content='What is 5+7?', additional_kwargs={}, response_metadata={}, id='64052d77-4e2a-477c-804c-097112af73e2'), AIMessage(content='5 + 7 equals 12.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 14, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_94cd41efba', 'id': 'chatcmpl-E0s9bvihrVTV7Lg53Ll3zzMIOkoH5', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f5758-d639-7ed0-b127-d80246a89bb4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 8, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'au

In [30]:
from langchain.tools import tool
@tool 
def search_hotels(city:str)->str:
    """Search hotal in city-return long use more tokens"""
    return f""" Hotel in {city}:
    1. Grand Hotel: 5 star hotal
    2.City Hotal: 4 star hotal
    3.Budget Hotal: 3 star hotal"""


agent =create_agent(  model="groq:qwen/qwen3-32b", 
    checkpointer=InMemorySaver(),
    tools=[search_hotels] , 
    middleware=[
        SummarizationMiddleware(
        model="groq:qwen/qwen3-32b",
        trigger=("tokens", 450),
        keep=("messages", 200)
        )
    ])

In [31]:
def countToken(messages):
    total_char = sum(len(str(n.content)) for n in messages)
    return total_char//4 

In [ ]:
config = {"configurable": {"thread_id": "test-2"}}

In [32]:
cities = ["Tokyo", "Paris", "New York", "London", "Sydney", "Cairo"]

for city in cities :
    res= agent.invoke({"messages":[HumanMessage(f"find hotels in {city}")]}, config=config)
    print(f"Messages: {res["messages"]} ")
    print(f"{city}: ~{countToken(res["messages"])} tokens,{len(res["messages"])} messages")

    

Messages: [HumanMessage(content='find hotels in Tokyo', additional_kwargs={}, response_metadata={}, id='bb34e3f5-114b-4cd2-b3b6-fe417fc4d6e2'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to find hotels in Tokyo. Let me check the available tools. There's a function called search_hotels that takes a city parameter. The description says it returns long results using more tokens. Since the user specified Tokyo, I need to call this function with the city set to Tokyo. I'll make sure the parameters are correctly formatted as JSON within the tool_call tags.\n", 'tool_calls': [{'id': 'nz4f57fq6', 'function': {'arguments': '{"city":"Tokyo"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 155, 'total_tokens': 260, 'completion_time': 0.169463434, 'completion_tokens_details': {'reasoning_tokens': 79}, 'prompt_time': 0.006168745, 'prompt_tokens_details': None, 'queue_time': 0.00